# Nvidia NemoRetriever + FAISS 索引示例

本notebook演示如何使用FAISS向量索引加速大规模文档检索

## 1. 环境准备和导入

In [1]:
%load_ext autoreload
%autoreload 2

# 加载 autoreload 扩展
# 设置自动重新加载模式。2 表示任何已导入的模块，只要其源代码发生变化，就会在执行下一行代码时自动重新加载

In [ ]:
import os
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
os.environ["TOKENIZERS_PARALLELISM"] = "true"
import torch
from pdf2image import convert_from_path
from colQwen_rag_with_faiss_pdf import MultiRoutePDFPipeline, GPUMemoryMonitor

colqwen_path = "tsystems/colqwen2.5-3b-multilingual-v1.0-merged"
gme_path = "Alibaba-NLP/gme-Qwen2-VL-7B-Instruct"
# 设置GPU设备
DEVICE = 0
torch.cuda.set_device(DEVICE)

print("✓ 环境准备完成")


## 2. 加载Retriever模型

In [ ]:
memory_monitor = GPUMemoryMonitor(DEVICE)
memory_monitor.print_memory("Before loading model")

# 加载Nvidia NemoRetriever模型
multi_route_pipeline = MultiRoutePDFPipeline(
    device_index=DEVICE,
    colqwen_path=colqwen_path,
    gme_path=gme_path,
)

memory_monitor.print_memory("After loading model")
print("✓ Retriever模型加载完成")


## 3. 加载PDF文档

In [3]:
# 加载PDF
pdf_path = 'contents/2024_Tencent_ESG.pdf'
images = convert_from_path(pdf_path, dpi=200)

print(f"✓ PDF加载完成，共 {len(images)} 页")

✓ PDF加载完成，共 111 页


## 4. 初始化RAG Pipeline（带FAISS索引）

In [ ]:
import numpy as np
# FAISS配置
# 初始化Pipeline
multi_route_pipeline.build_index(images)

print("✓ RAG Pipeline初始化完成")


✓ RAG Pipeline初始化完成


## 5. 编码文档并构建FAISS索引

In [ ]:
# 编码文档
# 在 MultiRoutePDFPipeline 中，build_index 已经完成必要的编码和缓存
print(f"✓ 文档编码完成, 共 {len(images)} 页")


In [ ]:
# 构建FAISS索引
# rag_pipeline.build_index(save_dir="./faiss_index/tencent_esg")
# print("✓ FAISS索引构建完成")

print("✓ FAISS索引加载完成")


## 6. 使用FAISS索引进行检索

In [ ]:
# 定义查询
queries = [
    '2022年员工总数是多少？',
    '以2021年作为基准年，2023年温室气体范围3排放量是否达到了减少的目标？',
    '截至二零二四年每收入单位的温室气体排放总量是多少？'
]

# 批量检索（统一使用 queries 参数）
import time

print("\n" + "="*80)
print("多路召回检索（ColQwen + GME Layout + GME Text）")
print("="*80)

start_time = time.time()

# 批量检索 - 传入查询列表
all_results = multi_route_pipeline.retrieve(
    queries=queries,
    top_k_colqwen=20,
    top_k_layout=20,
    top_k_text=20,
    final_top_k=15,
    rerank_colqwen=True,
)

elapsed_time = time.time() - start_time

# 显示结果 - all_results 是 List[Dict]
for i, (query, result_dict) in enumerate(zip(queries, all_results)):
    merged = result_dict["merged_results"]
    print(f"\n查询 {i+1}: {query}")
    print("-"*80)
    print("Top-15 融合结果:")
    for rank, item in enumerate(merged[:15], 1):
        print(
            f"  {rank}. 第 {item['page_num']} 页，"
            f"融合分数: {item['final_score']:.4f}, "
            f"ColQwen: {item['colqwen_score']:.4f}, "
            f"GME-Layout: {item['gme_layout_score']:.4f}, "
            f"GME-Text: {item['gme_text_score']:.4f}"
        )

print(f"\n总检索时间: {elapsed_time:.3f}秒 (平均每个查询 {elapsed_time/len(queries):.3f}秒)")


## 7. 加载ReRanker模型

In [ ]:
memory_monitor.print_memory("Before loading reranker")

# 初始化 Reranker
# 在 MultiRoutePDFPipeline 中，已经集成了 MonoVLM 图像重排序，这里不再加载单独的图像 Reranker。

memory_monitor.print_memory("After loading reranker")
print("✓ Reranker 加载完成（由 MonoVLM 在 Pipeline 内部完成）")


## 8. 批量 Reranking

In [ ]:
print("\n" + "="*80)
print("基于多路融合结果选择 Top-5 页面")
print("="*80)

# 准备每个查询的 Top-5 图片
all_top5_images = []
for i, result_dict in enumerate(all_results):
    merged = result_dict["merged_results"]
    top5 = merged[:5]
    images_i = [images[item["page_idx"]] for item in top5]
    all_top5_images.append(images_i)

    print(f"\n查询 {i+1}: {queries[i]}")
    print("Top-5 页面:")
    for rank, item in enumerate(top5, 1):
        print(f"  {rank}. 第 {item['page_num']} 页 (融合分数: {item['final_score']:.4f})")

print("\n✓ Top-5 页面选择完成")


In [ ]:
# all_top5_images 已在上一单元生成，这里无需重复处理


In [ ]:
import importlib
import colQwen_rag_with_faiss_pdf  # 需要重新导入模块本身

# 使用 reload 函数重新加载整个模块
importlib.reload(colQwen_rag_with_faiss_pdf)

# 重新从模块中获取更新后的类/函数
from colQwen_rag_with_faiss_pdf import VQAModel
# 初始化 VQA
vqa_model = VQAModel(
    model_name="doubao-seed-1-6-vision-250815"
)


✓ VQA Model initialized with API: doubao-seed-1-6-vision-250815


In [ ]:
# 批量问答（每个查询使用 Top-5 图片）
answers = vqa_model.answer_batch_with_multiple_images(
    queries=queries,
    all_images_list=all_top5_images,
    max_tokens=512
)

# 显示结果
for i, (query, answer) in enumerate(zip(queries, answers)):
    print(f"\n{'='*80}")
    print(f"问题 {i+1}: {query}")
    print(f"{'='*80}")
    print("使用页面: Top-5 综合分析")
    merged = all_results[i]["merged_results"]
    top5 = merged[:5]
    for rank, item in enumerate(top5, 1):
        print(f"  {rank}. 第 {item['page_num']} 页")
    print(f"\n答案:\n{answer}")
